# Air Quality Prediction and Respiratory Health Impact Modeling
## Modeling Phase

This notebook implements the **predictive modeling phase** following the exploratory data analysis (EDA).

Two modeling tasks are performed:

---

## Model 1 — AQI Forecast Model

Forecast future **population-weighted statewide AQI** using weekly time series data.

Dataset: `weekly_statewide_aqi.csv`

Models tested:
- **ARIMA** (baseline time-series model)
- **SARIMA** (seasonal ARIMA)
- **Random Forest** (time-series features)

---

## Model 2 — Lagged Respiratory Health Impact Model

Predict asthma health outcomes using air pollution exposure.

Dataset: `joint_aqi_health_county.csv`

Models tested:
- Linear Regression
- Random Forest Regression
- Gradient Boosting Regression

Targets:
- Asthma hospitalization rate
- Asthma ED visit rate

---

## Modeling Workflow

1. Load modeling datasets
2. Feature engineering
3. Train/test split
4. Train multiple models
5. Evaluate performance
6. Interpret feature importance
7. Export results to CSV

In [1]:
# ==========================================================
# Setup & Imports
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12,5)

DATA_DIR = Path("./data/outputs")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ==========================================================
# Import utility functions from src/ modules
# ==========================================================

import sys
sys.path.insert(0, ".")

from src.evaluation import evaluate_model, build_aqi_metrics_table, build_health_metrics_table
from src.modeling import chronological_split, fit_arima, fit_sarima, fit_rf_timeseries
from src.feature_engineering import add_grouped_lag, encode_county
from src.visualization import plot_forecast_comparison, plot_feature_importance

print("src modules loaded successfully.")

ModuleNotFoundError: No module named 'src'

# Model 1 — AQI Forecasting

The first model predicts **future statewide AQI levels**.

Key variable: **PopWeighted_AQI** — Population-weighted statewide AQI.

Steps:
1. Load weekly AQI time series
2. Visualize trend
3. Train ARIMA, SARIMA, and Random Forest models
4. Evaluate forecast accuracy (MAE, RMSE, R²)

In [ ]:
# ==========================================================
# Load Weekly AQI Dataset
# ==========================================================

aqi_weekly = pd.read_csv(DATA_DIR / "weekly_statewide_aqi.csv")
aqi_weekly["WeekEnd"] = pd.to_datetime(aqi_weekly["WeekEnd"])
aqi_weekly = aqi_weekly.sort_values("WeekEnd")

print("Dataset shape:", aqi_weekly.shape)
aqi_weekly.head()

In [ ]:
# ==========================================================
# Plot AQI Time Series
# ==========================================================

plt.plot(aqi_weekly["WeekEnd"], aqi_weekly["PopWeighted_AQI"])
plt.title("Weekly Population-Weighted AQI — California")
plt.ylabel("AQI")
plt.xlabel("Date")
plt.show()

## Train / Test Split

For time-series forecasting, the dataset is split chronologically.

- Training data: 2020–2023
- Test data: 2024

In [ ]:
# ==========================================================
# Train/Test Split
# ==========================================================

train_size = int(len(aqi_weekly) * 0.8)
train = aqi_weekly.iloc[:train_size]
test = aqi_weekly.iloc[train_size:].copy()

print("Train size:", train.shape)
print("Test size:", test.shape)

In [ ]:
# ==========================================================
# ARIMA Baseline Model
# ==========================================================

arima_model = ARIMA(train["PopWeighted_AQI"], order=(1,1,1))
arima_results = arima_model.fit()
arima_forecast = arima_results.forecast(steps=len(test))
test["ARIMA_Forecast"] = arima_forecast.values

In [ ]:
# ==========================================================
# SARIMA Model
# ==========================================================

sarima_model = SARIMAX(
    train["PopWeighted_AQI"],
    order=(1,1,1),
    seasonal_order=(1,1,1,52)
)
sarima_results = sarima_model.fit()
print(sarima_results.summary())

In [ ]:
# ==========================================================
# SARIMA Diagnostics
# ==========================================================

sarima_results.plot_diagnostics(figsize=(10,6))
plt.show()

In [ ]:
# ==========================================================
# SARIMA Forecast
# ==========================================================

sarima_forecast = sarima_results.forecast(steps=len(test))
test["SARIMA_Forecast"] = sarima_forecast.values

In [ ]:
# ==========================================================
# Random Forest Time-Series Model
# ==========================================================

aqi_weekly["lag1"] = aqi_weekly["PopWeighted_AQI"].shift(1)
aqi_weekly["lag2"] = aqi_weekly["PopWeighted_AQI"].shift(2)
aqi_weekly["lag3"] = aqi_weekly["PopWeighted_AQI"].shift(3)

rf_data = aqi_weekly.dropna()
rf_train = rf_data.iloc[:train_size]
rf_test = rf_data.iloc[train_size:].copy()

features = ["lag1","lag2","lag3"]

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(rf_train[features], rf_train["PopWeighted_AQI"])
rf_preds = rf.predict(rf_test[features])
rf_test["RF_Forecast"] = rf_preds

In [ ]:
# ==========================================================
# Forecast Visualization
# ==========================================================

plt.figure(figsize=(12,5))
plt.plot(train["WeekEnd"], train["PopWeighted_AQI"], label="Train")
plt.plot(test["WeekEnd"], test["PopWeighted_AQI"], label="Actual")
plt.plot(test["WeekEnd"], test["ARIMA_Forecast"], label="ARIMA")
plt.plot(test["WeekEnd"], test["SARIMA_Forecast"], label="SARIMA")
plt.plot(rf_test["WeekEnd"], rf_test["RF_Forecast"], label="Random Forest")
plt.legend()
plt.title("AQI Forecast Model Comparison")
plt.xlabel("Date")
plt.ylabel("AQI")
plt.show()

In [ ]:
# ==========================================================
# evaluate_model is imported from src/evaluation.py
# It returns (MAE, RMSE, R2) for any set of predictions.
# ==========================================================

arima_mae,  arima_rmse,  arima_r2  = evaluate_model(test["PopWeighted_AQI"], test["ARIMA_Forecast"])
sarima_mae, sarima_rmse, sarima_r2 = evaluate_model(test["PopWeighted_AQI"], test["SARIMA_Forecast"])
rf_mae,     rf_rmse,     rf_r2     = evaluate_model(rf_test["PopWeighted_AQI"], rf_test["RF_Forecast"])

results_df = build_aqi_metrics_table(
    model_names=["ARIMA", "SARIMA", "Random Forest"],
    true_vals=test["PopWeighted_AQI"],
    pred_lists=[test["ARIMA_Forecast"], test["SARIMA_Forecast"], rf_test["RF_Forecast"]]
)

print(results_df)

In [ ]:
# ==========================================================
# FIX 2: Export Model 1 results to CSV
# ==========================================================

# Export AQI model metrics
results_df.to_csv(DATA_DIR / "aqi_model_metrics.csv", index=False)
print("Saved: aqi_model_metrics.csv")

# Build and export forecast results
forecast_export = test[["WeekEnd", "PopWeighted_AQI", "ARIMA_Forecast", "SARIMA_Forecast"]].copy()
forecast_export = forecast_export.rename(columns={"PopWeighted_AQI": "Actual"})

# Merge Random Forest forecasts
rf_export = rf_test[["WeekEnd", "RF_Forecast"]].rename(columns={"RF_Forecast": "RandomForest"})
forecast_export = forecast_export.merge(rf_export, on="WeekEnd", how="left")

forecast_export.to_csv(DATA_DIR / "aqi_forecast_results.csv", index=False)
print("Saved: aqi_forecast_results.csv")

# Model 2 — Lagged Health Impact Model

The second model estimates how air pollution affects respiratory health outcomes.

Dataset: `joint_aqi_health_county.csv`

Features:
- Mean AQI
- AQI lag (previous year)
- County (encoded)
- Year

Targets:
1. Age-adjusted hospitalization rate
2. Age-adjusted ED visit rate

In [ ]:
# ==========================================================
# Load Joint Dataset
# ==========================================================

health = pd.read_csv(DATA_DIR / "joint_aqi_health_county.csv")
print("Dataset shape:", health.shape)
health.head()

In [ ]:
# ==========================================================
# FIX 3: Feature Engineering — Added County and Year features
# ==========================================================

health = health.sort_values(["County_join", "Year"])
health["AQI_lag1"] = health.groupby("County_join")["Mean_AQI"].shift(1)
health = health.dropna()

# Encode County as numeric
le = LabelEncoder()
health["County_encoded"] = le.fit_transform(health["County_join"])

print("Shape after feature engineering:", health.shape)
health.head()

In [ ]:
# ==========================================================
# FIX 3 continued: Features now include County and Year
# ==========================================================

features = ["Mean_AQI", "AQI_lag1", "County_encoded", "Year"]

In [ ]:
# ==========================================================
# Model 2a — Target: Hospitalization Rate
# ==========================================================

target_hosp = "AGE-ADJUSTED HOSPITALIZATION RATE"

df_hosp = health.dropna(subset=[target_hosp])
X_hosp = df_hosp[features]
y_hosp = df_hosp[target_hosp]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_hosp, y_hosp, test_size=0.2, random_state=42
)

# Linear Regression
lr_h = LinearRegression()
lr_h.fit(X_train_h, y_train_h)
pred_lr_h = lr_h.predict(X_test_h)

# Random Forest
rf_h = RandomForestRegressor(n_estimators=300, random_state=42)
rf_h.fit(X_train_h, y_train_h)
pred_rf_h = rf_h.predict(X_test_h)

# Gradient Boosting
gb_h = GradientBoostingRegressor(random_state=42)
gb_h.fit(X_train_h, y_train_h)
pred_gb_h = gb_h.predict(X_test_h)

print("--- Hospitalization Rate ---")
for name, preds in [("Linear Regression", pred_lr_h), ("Random Forest", pred_rf_h), ("Gradient Boosting", pred_gb_h)]:
    mae, rmse, r2 = evaluate_model(y_test_h, preds)
    print(f"{name}: R2={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")

In [ ]:
# ==========================================================
# FIX 4: Model 2b — Target: ED Visit Rate (was missing)
# ==========================================================

target_ed = "AGE-ADJUSTED ED VISIT RATE"

df_ed = health.dropna(subset=[target_ed])
X_ed = df_ed[features]
y_ed = df_ed[target_ed]

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_ed, y_ed, test_size=0.2, random_state=42
)

# Linear Regression
lr_e = LinearRegression()
lr_e.fit(X_train_e, y_train_e)
pred_lr_e = lr_e.predict(X_test_e)

# Random Forest
rf_e = RandomForestRegressor(n_estimators=300, random_state=42)
rf_e.fit(X_train_e, y_train_e)
pred_rf_e = rf_e.predict(X_test_e)

# Gradient Boosting
gb_e = GradientBoostingRegressor(random_state=42)
gb_e.fit(X_train_e, y_train_e)
pred_gb_e = gb_e.predict(X_test_e)

print("--- ED Visit Rate ---")
for name, preds in [("Linear Regression", pred_lr_e), ("Random Forest", pred_rf_e), ("Gradient Boosting", pred_gb_e)]:
    mae, rmse, r2 = evaluate_model(y_test_e, preds)
    print(f"{name}: R2={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")

In [ ]:
# ==========================================================
# Feature Importance (Hospitalization)
# ==========================================================

importance = pd.Series(rf_h.feature_importances_, index=features)
importance.sort_values().plot(kind="barh")
plt.title("Feature Importance — Hospitalization Model")
plt.show()

In [ ]:
# ==========================================================
# FIX 2 continued: Export Model 2 results to CSV
# ==========================================================

# Build health model metrics table (both targets)
health_metrics_rows = []
for name, pred_h, pred_e in [
    ("Linear Regression", pred_lr_h, pred_lr_e),
    ("Random Forest", pred_rf_h, pred_rf_e),
    ("Gradient Boosting", pred_gb_h, pred_gb_e)
]:
    mae_h, rmse_h, r2_h = evaluate_model(y_test_h, pred_h)
    mae_e, rmse_e, r2_e = evaluate_model(y_test_e, pred_e)
    health_metrics_rows.append({
        "Model": name,
        "Hosp_MAE": round(mae_h, 4), "Hosp_RMSE": round(rmse_h, 4), "Hosp_R2": round(r2_h, 4),
        "ED_MAE": round(mae_e, 4), "ED_RMSE": round(rmse_e, 4), "ED_R2": round(r2_e, 4)
    })

health_metrics_df = pd.DataFrame(health_metrics_rows)
health_metrics_df.to_csv(DATA_DIR / "health_model_metrics.csv", index=False)
print("Saved: health_model_metrics.csv")
print(health_metrics_df)

# Build and export health predictions
health_preds_df = X_test_h.copy()
health_preds_df["Actual"] = y_test_h.values
health_preds_df["Pred_LR"] = pred_lr_h
health_preds_df["Pred_RF"] = pred_rf_h
health_preds_df["Pred_GB"] = pred_gb_h
health_preds_df.to_csv(DATA_DIR / "health_model_predictions.csv", index=False)
print("Saved: health_model_predictions.csv")

## Conclusion

This modeling phase developed two predictive models to analyze the relationship between air quality and respiratory health outcomes in California.

The first model applied time-series forecasting techniques to predict weekly population-weighted AQI levels. Three models were evaluated: ARIMA, SARIMA, and a Random Forest regression model using lagged AQI features. Among these models, the SARIMA model achieved the lowest forecasting error, with a mean absolute error of approximately 12 AQI points. The Random Forest model produced very similar performance, while the baseline ARIMA model showed substantially higher error. These results indicate that incorporating seasonal patterns improves AQI forecasting accuracy.

The second model examined the relationship between air pollution exposure and respiratory health outcomes using county-level data. Both asthma hospitalization rate and ED visit rate were modeled as separate targets. Lagged AQI, county, and year were included as features to capture delayed health effects and regional variation. Linear regression, Random Forest, and Gradient Boosting models were evaluated.

Feature importance analysis showed that county and year were the most influential predictors, with AQI and lagged AQI contributing additional explanatory power. This is consistent with environmental health research, which shows that respiratory outcomes are influenced by multiple factors such as particulate matter concentration, weather conditions, wildfire activity, and socioeconomic factors.

Overall, the modeling results provide quantitative evidence that air quality trends can be forecasted using historical data and that air pollution is associated with respiratory health impacts, although additional environmental and demographic variables are likely required to fully explain variation in asthma hospitalization and ED visit rates.